# Solution 6 — UR5e kinematics checkpoint

Chapter 17 contains the full OCP; this solution validates the kinematic derivative used by its obstacle constraints.


In [ ]:
# Locate the repository so these editable modules work without installation.
import sys
from pathlib import Path

repository_root = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "casadi_tutorial").is_dir()
)
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

# Reload edited model modules before each cell during tutorial development.
ipython = get_ipython()
if ipython is not None:
    ipython.run_line_magic("load_ext", "autoreload")
    ipython.run_line_magic("autoreload", "2")

In [ ]:
import casadi as ca
import numpy as np

from casadi_tutorial.models.ur5e import (
    forward_kinematics,
    geometric_jacobian,
)

joint_vector = ca.SX.sym("joint_vector", 6)
tool_position = ca.Function(
    "exercise_tool_position",
    [joint_vector],
    [forward_kinematics(joint_vector)[:3, 3]],
)
translational_jacobian = ca.Function(
    "exercise_jacobian",
    [joint_vector],
    [geometric_jacobian(joint_vector)[:3, :]],
)

# Validate the symbolic Jacobian with a centered directional difference.
configuration = np.array([0.1, -0.4, 0.6, -0.2, 0.3, 0.1])
direction = np.arange(1, 7) / 10
epsilon = 1e-6
finite_difference = (
    tool_position(configuration + epsilon * direction)
    - tool_position(configuration - epsilon * direction)
) / (2 * epsilon)
np.testing.assert_allclose(
    finite_difference.full().ravel(),
    (translational_jacobian(configuration) @ direction).full().ravel(),
    rtol=1e-6,
    atol=1e-8,
)